### 05.구조화된 출력 파서 StructuredOutputParser page179
- 키-값(Key-Value) JSON 딕셔너리(dict)로 파싱
- ResponseSchema 기반 정의
- 프롬프트 자동 지시사항 주입
- 파이썬 dict 반환
- Pydantic 또는 TypedDict 사용.
- 파싱기호 누락 등의 파싱 에러가 있을 수 있음.

In [1]:
# !pip --version

In [2]:
# !pip install dotenv
from dotenv import load_dotenv

# .env파일에 설정된 보안정보를 읽기.
load_dotenv()

False

In [3]:
!pip install langchain_classic


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# 오류 ModuleNotFoundError
# from output_parsers.output_parsers import ResponseSchema, StructuredOutputParser
# 수정제안
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [5]:
# 사용자의 질문에 대한 답변
response_schemas = [
    ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
    ResponseSchema(
        name="source",
        description="사용자의 질문에 답하기 위해 사용된 `출처`, `웹사이트주소` 이여야 합니다.",
    ),
]

In [6]:
# 응답 스키마를 기반으로 한 구조화된 출력 파서 초기화
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [7]:
print(output_parser.get_format_instructions())

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"answer": string  // 사용자의 질문에 대한 답변
	"source": string  // 사용자의 질문에 답하기 위해 사용된 `출처`, `웹사이트주소` 이여야 합니다.
}
```


In [8]:
# 출력 형식 지시사항을 파싱합니다.
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    # 사용자의 질문에 최대한 답변하도록 템플릿을 설정합니다.
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",
    # 입력 변수로 'question'을 사용합니다.
    input_variables=["question"],
    # 부분 변수로 'format_instructions'을 사용합니다.
    partial_variables={"format_instructions": format_instructions},
)

In [9]:
model = ChatOpenAI(temperature=0)  # ChatOpenAI 모델 초기화
chain = prompt | model | output_parser  # 프롬프트, 모델, 출력 파서를 연결

In [10]:
chain.invoke({"question": "아르헨티나의 수도는 어디인가요?"})

{'answer': '아르헨티나의 수도는 부에노스아이레스입니다.',
 'source': 'https://ko.wikipedia.org/wiki/%EC%95%84%EB%A5%B4%EA%B2%A8%ED%8B%80%EB%8B%88%EC%95%84'}

In [11]:
for s in chain.stream({"question": "세종대왕의 업적은 무엇인가요?"}):
    # 스트리밍 출력
    print(s)

{'answer': '세종대왕은 한글을 창제하고 문화를 발전시키는 등 다양한 업적을 남겼습니다.', 'source': 'https://ko.wikipedia.org/wiki/%EC%84%B8%EC%A2%85%EB%8C%80%EC%99%95'}


### JSON 형식 출력 파서

In [12]:
'''json
{
    "name": "John Doe",
    "age": 30,
    "is_student": false,
    "skills": ["Java", "Python", "JavaScript"],
    "address": {
        "street": "123 Main St",
        "city": "Anytown"
    }
}
'''

'json\n{\n    "name": "John Doe",\n    "age": 30,\n    "is_student": false,\n    "skills": ["Java", "Python", "JavaScript"],\n    "address": {\n        "street": "123 Main St",\n        "city": "Anytown"\n    }\n}\n'

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser")

model = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser


In [4]:
class Topic(BaseModel): # 원하는 데이터 구조를 정의
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [5]:
question = "지구 온난화의 심각성에 대해 알려주세요."

parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [7]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser # 체인을 구성

answer = chain.invoke({"question": question}) # 체인을 호출

In [8]:
answer["description"]

'지구 온난화는 기후 변화의 주요 원인으로, 지구의 평균 기온이 상승하여 극단적인 기상 현상, 해수면 상승, 생태계 파괴 등을 초래하고 있습니다.'